In [ ]:
!mkdir ~/tmpdir
!TMPDIR=~/tmpdir python3 -m pip install --upgrade --no-cache-dir "sagemaker<3" pandas numpy matplotlib seaborn scikit-learn "torch==2.6" torchvision boto3

In [ ]:
!curl -L <URL> -o dataset.zip
!unzip -o dataset.zip -d dataset
!find dataset -type f

In [ ]:
import pandas as pd
import numpy as np
import torch
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from IPython.display import display
from sagemaker.inputs import TrainingInput
from sagemaker.s3 import S3Uploader

In [ ]:
!rm -f dataset.zip
%cd dataset
!zip -rq ../dataset.zip .
%cd ..

import sagemaker
from sagemaker import get_execution_role
from sagemaker.s3 import S3Uploader

session = sagemaker.Session()
bucket = session.default_bucket()
role = get_execution_role()

data_s3 = S3Uploader.upload(
    "dataset.zip",
    f"s3://{bucket}/sagemaker/dataset/",
)
print(data_s3)

In [ ]:
_dataset = datasets.ImageFolder(root='./dataset', transform=transforms.ToTensor())
_loader  = DataLoader(_dataset)

mean   = torch.zeros(3)
sq_mean = torch.zeros(3)
n_pixels = 0

with torch.no_grad():
    for images, _ in _loader:
        b, c, h, w = images.shape
        n_pixels += b * h * w
        mean     += images.sum(dim=[0, 2, 3])
        sq_mean  += (images ** 2).sum(dim=[0, 2, 3])

mean  /= n_pixels
std    = (sq_mean / n_pixels - mean ** 2).sqrt()

mean, std

In [ ]:
def compression(args):
    with zipfile.ZipFile(f'{args.train}/dataset.zip', 'r') as zip_ref:
        zip_ref.extractall(f'{args.train}/')

    logger.info("====== Dataset loaded ======")

In [ ]:
def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

In [ ]:
from PIL import Image

img = Image.open("image.jpg")
print(img.size)